In [2]:
import pandas as pd
import sqlite3
from IPython.display import display

# 1. Boot up the in-memory SQL database
conn = sqlite3.connect(':memory:')
print("🟢 Booting up CloudSync SQL Server...")

# 2. Load the CSVs from your data folder into SQL tables
try:
    pd.read_csv('data/saas_customers.csv').to_sql('customers', conn, index=False, if_exists='replace')
    pd.read_csv('data/saas_engagement.csv').to_sql('engagement', conn, index=False, if_exists='replace')
    print("✅ SUCCESS: SQL Sandbox is ready!\n")
except FileNotFoundError:
    print("🚨 ERROR: Check your file paths!")

🟢 Booting up CloudSync SQL Server...
✅ SUCCESS: SQL Sandbox is ready!



In [3]:
# ----------------------------------------------------
# QUERY 2: The Root Cause - Investigating the 'Pro' Tier
# ----------------------------------------------------
query_2 = """
SELECT 
    c.subscription_tier,
    CASE 
        WHEN e.support_tickets_last_30 >= 5 THEN 'High (5+ Tickets)'
        WHEN e.support_tickets_last_30 BETWEEN 1 AND 4 THEN 'Moderate (1-4 Tickets)'
        ELSE 'Zero Tickets' 
    END AS ticket_volume,
    COUNT(c.customer_id) AS total_users,
    ROUND((SUM(CASE WHEN c.churn_status = 'Churned' THEN 1.0 ELSE 0 END) / COUNT(c.customer_id)) * 100, 1) AS churn_rate_percentage
FROM 
    customers c
JOIN 
    engagement e ON c.customer_id = e.customer_id
WHERE 
    c.subscription_tier = 'Pro'
GROUP BY 
    c.subscription_tier,
    ticket_volume
ORDER BY 
    churn_rate_percentage DESC;
"""

# Run the SQL query and display the results
print("🔍 Analyzing Pro Tier Support Tickets vs. Churn Rate...")
result_2 = pd.read_sql(query_2, conn)
display(result_2)

🔍 Analyzing Pro Tier Support Tickets vs. Churn Rate...


,subscription_tier,ticket_volume,total_users,churn_rate_percentage
0,Pro,High (5+ Tickets),659,44.5
1,Pro,Moderate (1-4 Tickets),1044,17.2


In [4]:
# ----------------------------------------------------
# QUERY 3: Calculating Lost MRR (Monthly Recurring Revenue)
# ----------------------------------------------------
query_3 = """
SELECT 
    subscription_tier,
    COUNT(customer_id) AS total_churned_users,
    SUM(mrr) AS lost_monthly_revenue
FROM 
    customers
WHERE 
    churn_status = 'Churned'
GROUP BY 
    subscription_tier
ORDER BY 
    lost_monthly_revenue DESC;
"""

# Run the SQL query and display the results
print("💰 Calculating Revenue Lost to Churn...")
result_3 = pd.read_sql(query_3, conn)

# Format the revenue column to look like currency ($)
result_3['lost_monthly_revenue'] = result_3['lost_monthly_revenue'].apply(lambda x: f"${x:,.2f}")
display(result_3)

💰 Calculating Revenue Lost to Churn...


,subscription_tier,total_churned_users,lost_monthly_revenue
0,Pro,473,"$23,177.00"
1,Enterprise,39,"$7,761.00"
2,Basic,252,"$4,788.00"
